# 3 — Release and change

*The SemOps Manual — Chapter 12*

Turning "is this a breaking change?" from an opinion into a command, and letting
the mechanical half of a migration apply itself.

---

## Prerequisites

This notebook is **self-contained**: it writes its own fixtures into a temporary
directory, so nothing needs to exist on disk beforehand. What it does need:

```bash
pip install ontology-quality-suite shacl
```

The next cell checks what is available and prints exactly what is missing.
Cells that need a tool you do not have will say so and skip, rather than
failing with a stack trace.

In [ ]:
import json, os, shutil, subprocess, sys, tempfile, textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="semops-nb-"))
print("working directory:", WORK)


def have(mod):
    try:
        __import__(mod)
        return True
    except ImportError:
        return False


HAVE_SUITE = have("ontology_suite")
HAVE_SHACL = have("shacl")
SHACL_CLI = shutil.which("shacl")

print("ontology-quality-suite:", "yes" if HAVE_SUITE else "NO  (pip install ontology-quality-suite)")
print("shacl (python)       :", "yes" if HAVE_SHACL else "NO  (pip install shacl)")
print("shacl (cli)          :", SHACL_CLI or "not on PATH")

if HAVE_SHACL:
    import shacl as shacl_py
    print("shacl version        :", getattr(shacl_py, "__version__", "unknown"))


def write(name, text):
    """Write a fixture into the working directory and return its path."""
    p = WORK / name
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(textwrap.dedent(text).lstrip(), encoding="utf-8")
    return p


def suite(*args):
    """Run the ontology suite CLI and show its output."""
    if not HAVE_SUITE:
        print("skipped: ontology-quality-suite is not installed")
        return None
    r = subprocess.run([sys.executable, "-m", "ontology_suite", *map(str, args)],
                       capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip()[:2000])
    return r

## Two versions of an ontology

v2 does what a real release does: renames one term, adds another, and leaves the
existing flaws alone. The rename carries a **migration annotation**, which is what
lets tooling recognise it as a rename rather than a deletion plus an addition.

In [ ]:
v1 = write("acme-v1.ttl", """
    @prefix owl:  <http://www.w3.org/2002/07/owl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix acme: <https://acme.example.org/ns/> .

    <https://acme.example.org/ns/> a owl:Ontology ; owl:versionInfo "1.0.0" .
    acme:Employee a owl:Class ; rdfs:label "Employee" .
    acme:Engineer a owl:Class ; rdfs:subClassOf acme:Employee ; rdfs:label "Engineer" .
""")

v2 = write("acme-v2.ttl", """
    @prefix owl:  <http://www.w3.org/2002/07/owl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix acme: <https://acme.example.org/ns/> .

    <https://acme.example.org/ns/> a owl:Ontology ; owl:versionInfo "2.0.0" .
    acme:Employee a owl:Class ; rdfs:label "Employee" .

    # Engineer is retired in favour of SoftwareEngineer. The annotation is
    # asserted FROM the retiring IRI -- see below for why the direction matters.
    acme:Engineer owl:equivalentClass acme:SoftwareEngineer .
    acme:SoftwareEngineer a owl:Class ; rdfs:subClassOf acme:Employee ;
        rdfs:label "Software engineer" .

    acme:ProductManager a owl:Class ; rdfs:subClassOf acme:Employee ;
        rdfs:label "Product manager" .
""")
print("v1 and v2 written")

## Is it breaking?

The verdict comes from what happened to the axioms, not from what the author
believed they were doing. One removed class makes it MAJOR regardless of how
many additive changes surround it.

Note the argument order: **old first, then new**.

In [ ]:
r = suite("version-diff", v1, v2, "--exclude-imports",
          "--out-dir", WORK / "out/vdiff", "--json")

`--exclude-imports` is right here: `version-diff` compares your own axioms
against your own axioms, and merging identical copies of an upstream vocabulary
into both sides adds nothing. The next command needs the opposite — see below.

`--fail-on major` makes it enforceable; `--json` writes a machine-readable diff
a downstream partner can act on without parsing your release notes.

## What breaks, and can it fix itself?

A transformation query that types everything `acme:Engineer` is exactly what v2
breaks. `consistency` finds it and proposes a repair.

In [ ]:
query = write("queries/employees.rq", """
    PREFIX acme: <https://acme.example.org/ns/>

    # Every row is typed acme:Engineer -- the term v2 renames.
    CONSTRUCT {
      ?employee a acme:Employee, acme:Engineer .
    }
    WHERE {
      BIND(IRI(CONCAT("https://acme.example.org/data/employee/", ?id)) AS ?employee)
    }
""")

r = suite("consistency", "--new", v2, "--old", v1,
          "--queries", WORK / "queries",
          "--out-dir", WORK / "out/consistency")

### The direction of the migration annotation

`owl:equivalentClass` is logically symmetric — a reasoner treats both directions
identically. Rename **detection** does not: it reads the annotation from the
retiring IRI to its replacement. Written the other way round, confidence drops
to name-similarity guessing, silently.

This cell writes v2 with the annotation reversed, so you can see the difference.

In [ ]:
v2r = write("acme-v2-reversed.ttl",
            v2.read_text(encoding="utf-8").replace(
                "acme:Engineer owl:equivalentClass acme:SoftwareEngineer .",
                "acme:SoftwareEngineer owl:equivalentClass acme:Engineer ."))

r = suite("consistency", "--new", v2r, "--old", v1,
          "--queries", WORK / "queries",
          "--out-dir", WORK / "out/consistency-reversed")
print("\n^ compare the confidence with the run above.")

## Applying the repair

By default nothing is modified — repairs are written as reviewable `.patch` files.
`--apply-repairs` writes them in place, gated on confidence.

In [ ]:
before = query.read_text(encoding="utf-8")

r = suite("consistency", "--new", v2, "--old", v1,
          "--queries", WORK / "queries", "--apply-repairs",
          "--min-confidence", "0.7",
          "--out-dir", WORK / "out/applied")

after = query.read_text(encoding="utf-8")
print("\n--- what changed ---")
for b, a in zip(before.splitlines(), after.splitlines()):
    if b != a:
        print("  -", b.strip())
        print("  +", a.strip())

Look closely at the diff: the substitution is **textual across the whole file,**
**including comments**. The comment explaining that rows are typed `acme:Engineer`
now says `acme:SoftwareEngineer`, which makes it nonsense. Harmless — comments do
not execute — but it is the reason to prefer the default dry-run in automation
and review the patch, rather than letting an unattended job commit its own output.

---

## What to take away

- `version-diff` makes the bump **evidence-based**: the release conversation stops
  being about who is most confident.
- Assert migration annotations **from the retiring IRI**. Symmetric in logic,
  directional in tooling.
- Confidence-gate repairs: automate what is evidenced, escalate what is inferred.
- Resolve imports when a check reasons about terms you did not define; exclude
  them when it only compares terms you did.